In [1]:
import torch 
import torch.nn as nn 
import transformers 
from transformers import AutoModelForSequenceClassification,AutoTokenizer 

In [3]:
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [ ]:
Sequence = "The movie was a big loss."
tokenized_input = tokenizer.tokenize(Sequence)
input_ids =torch.tensor([tokenizer.convert_tokens_to_ids(tokenized_input)]) # batch first 
output = model(input_ids)
print(torch.nn.functional.softmax(output.logits,dim=-1).argmax(dim=1))

tensor([0])


# Processing data using Transformers

In [4]:
from datasets import load_dataset 
from transformers import AutoTokenizer
raw_data = load_dataset("glue","mrpc")
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [5]:
train_dataset = raw_data['train']
test_dataset = raw_data['test']

tokens = tokenizer(train_dataset[15]['sentence1'],train_dataset[15]['sentence2'])
print(tokens)


{'input_ids': [101, 24049, 2001, 2087, 3728, 3026, 3580, 2343, 2005, 1996, 9722, 1004, 4132, 9340, 12439, 2964, 2449, 1012, 102, 3026, 3580, 2343, 4388, 24049, 1010, 3839, 2132, 1997, 1996, 9722, 1998, 4132, 9340, 12439, 2964, 3131, 1010, 2097, 2599, 1996, 2047, 9178, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [6]:
def tokenize_function(example):
    return tokenizer(example['sentence1'],example['sentence2'],truncation=True)

In [9]:
tokenized_dataset = raw_data.map(tokenize_function,batched=True)
tokenized_dataset

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

In [7]:
from transformers import DataCollatorWithPadding 
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [10]:
samples = tokenized_dataset['train'][:8]
samples = {k:v for k,v in samples.items() if k not in ['idx','sentence1','sentence2']}
print([len(x) for x in samples['input_ids']]) # different length of different sentence after tokenization 
batch = data_collator(samples)
{k:v.shape for k,v in batch.items()}


[50, 59, 47, 67, 59, 50, 62, 32]


{'input_ids': torch.Size([8, 67]),
 'token_type_ids': torch.Size([8, 67]),
 'attention_mask': torch.Size([8, 67]),
 'labels': torch.Size([8])}

Using a Trainer to Create a complete pipeline

In [14]:
from transformers import AutoModelForSequenceClassification,AutoTokenizer,DataCollatorWithPadding 
from datasets import load_dataset 
checkpoint = "bert-base-uncased"

In [19]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint,num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
raw_data = load_dataset("glue","mrpc")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [20]:
def tokenize_function(sample):
    return tokenizer(sample['sentence1'],sample['sentence2'],truncation=True)

tokenized_dataset = raw_data.map(tokenize_function,batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

In [24]:
from transformers import TrainingArguments,Trainer
training_args = TrainingArguments('test-trainer',num_train_epochs=5)
trainer = Trainer(
    model,
    training_args,
    train_dataset = tokenized_dataset['train'],
    eval_dataset = tokenized_dataset['validation'],
    data_collator=data_collator,
    processing_class = tokenizer
)


In [25]:
trainer.train()

Step,Training Loss
500,0.367458
1000,0.240883
1500,0.138536
2000,0.066104


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2295, training_loss=0.1815124441076208, metrics={'train_runtime': 348.0652, 'train_samples_per_second': 52.691, 'train_steps_per_second': 6.594, 'total_flos': 675891190117440.0, 'train_loss': 0.1815124441076208, 'epoch': 5.0})

In [ ]:
prediction = model.predict(tokenized_dataset['validation'])
preds = prediction.argmax(dim=1)
print(preds)

In [27]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.8 MB/s eta 0:00:00


In [28]:
import evaluate 
import numpy as np
def compute_metrics(eval_preds):
    metrics = evaluate.load("glue","mrpc")
    logits,labels = eval_preds
    predictions = np.argmax(logits,axis=-1)
    return metrics.compute(predictions=predictions,references=labels)


In [30]:
training_arguments = TrainingArguments(
    "test-trainer",
    num_train_epochs=5,
    eval_strategy="epoch",
)

model = AutoModelForSequenceClassification.from_pretrained(checkpoint,num_labels=2)
trainer = Trainer(
    model,
    training_arguments,
    train_dataset = tokenized_dataset['train'],
    eval_dataset = tokenized_dataset['validation'],
    data_collator = data_collator,
    processing_class = tokenizer,
    compute_metrics = compute_metrics,
)

trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.506933,0.786765,0.858075
2,0.595696,0.434402,0.818627,0.865455
3,0.438423,0.609687,0.830882,0.875676
4,0.299786,0.777317,0.821078,0.873921
5,0.153178,0.930989,0.818627,0.871528


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2295, training_loss=0.33986894861026007, metrics={'train_runtime': 467.0995, 'train_samples_per_second': 39.264, 'train_steps_per_second': 4.913, 'total_flos': 675891190117440.0, 'train_loss': 0.33986894861026007, 'epoch': 5.0})

In [31]:
model.save_pretrained("/")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [34]:
save_directory = "Day-wise/Day-21/"
# Save both tokenizer and model to the local folder
tokenizer.save_pretrained(save_directory)
model.save_pretrained(save_directory)
print(f"Model and tokenizer saved to {save_directory}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved to Day-wise/Day-21/
